# AutoGluon
Matar una mosca con una  bazooka, entrena automaticamente modelos de:


*   Estadistica Clasica
*   Machine Learning ~ GBDT
*   Deep Learning


 ['SeasonalNaive', 'RecursiveTabular', 'DirectTabular', 'DynamicOptimizedTheta', 'Chronos2', 'Chronos2SmallFineTuned', 'AutoETS', 'ChronosWithRegressor[bolt_small]', 'TemporalFusionTransformer', 'DeepAR']



![Matar una mosca con una bazooka ](https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/Bazooka_fly.jpg)

## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [21]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Drive already mounted at /content/.drive; to attempt to forcibly remount, call drive.mount("/content/.drive", force_remount=True).


Para correr la siguiente celda es fundamental, POR UNICA VEZ, seguir los siguientes pasos

* Registrar usuario en Kaggle con la cuenta de email de la Universidad Austral
* Hacer el "Join Competition"  a la competencia de  Labo 3
* Generar el archivo kaggle.json  a partir de   https://www.kaggle.com/settings/account  y presione  "Create Legacy API Key"
* Crear carpeta  labo3  en  el Google Drive
* Dentro de la carpeta labo3 crear carpeta   kaggle
* Subir a la carpeta kaggle el archivo  kaggle.json


In [22]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"


cp: cannot stat '/content/buckets/b1/kaggle/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


# 1  Modelo AutoGluon

## 1.1 Init Experimento

In [23]:
# instalacion de paquetes que NO vienen por default en Colab
# autogluon es MUY pesado. varios minutos se perderan aqui
!pip install uv
!uv pip install -q kaggle
!uv pip install autogluon[all]


Using Python 3.12.13 environment at: /usr
Checked 1 package in 158ms


In [24]:
# funcion para hacer submits a Kaggle
def kaggle_submit(competencia, archivo, mensaje):

  # comando
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  # ejecucion
  os.system(comando)


In [25]:
import os as os
import numpy as np
import polars as pl

from datetime import datetime
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

Por favor, cargar aqui SU semilla primigenia
<br> **Muy importante**, cambiar el numero de experimento en cada corrida. Usted ha sido notificado !
<br> Si cada corrida no está en una nueva carpeta virgen, entonces se reutilizarn modelos viejos corridos con otros parametros
https://www.youtube.com/shorts/0_0Kzqpdn1o

In [26]:
# defino los parametros
PARAM = {'experimento':'AutoGluon-01',
  'kaggle_competition':'labo-iii-2026-ba',
  'semilla_primigenia':103000
}

In [27]:
# creo la carpeta del experimento y hago el chdir
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

/content/buckets/b1/exp/AutoGluon-01


## 1.2 Init AutoGluon

In [28]:
# cargo el dataset del sell-in
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [29]:
# agrupo por product_id, periodo
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)

tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [30]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")


# Filtro tb_ventas a solo las que debo predecir
print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir,
  on="product_id",
  how="inner"
)
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

31243
22349


In [31]:
# paso de periodo a  timestamp
tb_ventas = tb_ventas.with_columns(
    (pl.col('periodo').cast(pl.String).str.to_datetime('%Y%m')).alias('timestamp')
)

Opcion de *empiojar el dataset*
<br>agregando ruido relativo a las ventas
<br>Un Experimento no se le niega a nadie

In [32]:
empiojar= False
empiojar_ruido= 0.25

if empiojar:
  np.random.seed(PARAM['semilla_primigenia'])
  tb_ventas = tb_ventas.sort(["product_id", "periodo"])
  # vector con el ruido multiplicativo de media 1.0  y desvio  'empiojar_ruido'
  noise_multiplier = np.random.lognormal(mean=0.0, sigma=empiojar_ruido, size=tb_ventas.height)

  tb_ventas = tb_ventas.with_columns(
    (pl.col("tn") * pl.lit(noise_multiplier)).alias("tn")
  )


## 1.3 Entrenamiento AutoGluon

La magia del Auto Machine Learning  aplicada a un dataset que posee MULTIPLES series de tiempo que se analizan en forma conjunta, suponiendo algun tipo de correlacion entre ellas.

AutoGluon TimeSeriesPredictor predicts future values of **multiple related** time series.

TimeSeriesPredictor provides probabilistic (quantile) multi-step-ahead orecasts for univariate time series. The forecast includes both the mean (i.e.,
 onditional expectation of future values given the past), as well as the quantiles of the forecast distribution, indicating the range of possible future outcomes.

TimeSeriesPredictor fits both “global” deep learning models that are **shared across all time series** (e.g., DeepAR, Transformer), as well as “local”
 statistical models that are fit to each individual time series (e.g., ARIMA, ETS).

TimeSeriesPredictor expects input data and makes predictions in the TimeSeriesDataFrame format.


https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.html

In [33]:
# paso a formato  ts = time_series
ts_data = TimeSeriesDataFrame.from_data_frame(
  tb_ventas.to_pandas(), # tristemente AutoGluon espera un dataframe Pandas ...
  timestamp_column='timestamp',
  id_column='product_id'
)

Elegir cuidadosamente la metrica a utilizar
<br>Probar alternativas !
<br>Ese celda lleva 30 minutos en correr
<br>https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-metrics.html#forecasting-metrics
<br>https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.fit.html#autogluon.timeseries.TimeSeriesPredictor.fit
<br>https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.html

In [34]:
# Entrenamiento, esta es la parte pesada
global_eval_metric = 'RMSE' # alternativa RMSE

# defino
modelo = TimeSeriesPredictor(
  prediction_length= 2,  # horizonte de prediccion
  target= 'tn',
  freq= 'MS',  # Frecuencia mensual (Month Start)
  eval_metric= global_eval_metric
)

# entreno, le tira con muchisimos algorimtos, y prueba ensamblarlos
modelo.fit(ts_data,
  num_val_windows= 2,
  time_limit= 3600, # una hora
  presets= "best_quality",  # Máxima calidad posible, obvio mayor tiempo de corrida
  random_seed= PARAM['semilla_primigenia']
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260701_232520"
Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to '/content/.drive/My Drive/labo3/exp/AutoGluon-01/AutogluonModels/ag-20260701_232520'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
GPU Memory:         
Total GPU Memory:   Free: 0.00 GB, Allocated: 0.00 GB, Total: 0.00 GB
GPU Count:          0
Memory Avail:       10.06 GB / 12.67 GB (79.4%)
Disk Space Avail:   79.32 GB / 107.72 GB (73.6%)
Setting presets to: best_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': RMSE,
 'freq': 'MS',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 

## 1.4 Prediccion AutoGluon

In [40]:
# predict a partir los mismos datos

tb_forecast = modelo.predict(ts_data,
  random_seed= PARAM['semilla_primigenia']
)

display(tb_forecast)

data with frequency 'IRREG' has been resampled to frequency 'MS'.
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


mean         0.1          0.2          0.3  \
item_id timestamp                                                       
20001   2020-01-01  1285.685802  864.450144  1034.917432  1130.387515   
        2020-02-01  1297.394839  869.667369  1027.944816  1128.985652   
20002   2020-01-01  1222.060298  848.434844   981.657262  1086.128085   
        2020-02-01  1090.900843  694.109831   859.922531   948.347066   
20003   2020-01-01   838.788123  583.188352   672.345559   739.010546   
...                         ...         ...          ...          ...   
21266   2020-02-01     0.050203   -0.072745    -0.028717     0.001651   
21267   2020-01-01     0.029405   -0.036934    -0.010553     0.005449   
        2020-02-01     0.029524   -0.057321    -0.024509    -0.003068   
21276   2020-01-01     0.013389   -0.011471    -0.002803     0.003912   
        2020-02-01     0.013357   -0.022378    -0.008860    -0.001018   

                            0.4          0.5          0.6          0.7  \
item_id timestamp                                                        
20001   2020-01-01  1217.646852  1299.548199  1370.528267  1468.598767   
        2020-02-01  1227.931066  1312.852968  1395.318654  1478.483922   
20002   2020-01-01  1154.430465  1218.991183  1289.864782  1377.884755   
        2020-02-01  1015.103333  1083.291578  1169.095215  1255.669555   
20003   2020-01-01   783.656164   824.259579   881.655439   929.670431   
...                         ...          ...          ...          ...   
21266   2020-02-01     0.026670     0.051835     0.073954     0.100236   
21267   2020-01-01     0.018289     0.030674     0.042765     0.057284   
        2020-02-01     0.013863     0.031164     0.046392     0.062873   
21276   2020-01-01     0.008543     0.012984     0.018125     0.023183   
        2020-02-01     0.006148     0.012042     0.018867     0.025367   

                            0.8          0.9  
item_id timestamp                             
20001   2020-01-01  1553.305131  1692.643010  
        2020-02-01  1565.737880  1683.379971  
20002   2020-01-01  1473.170004  1614.483504  
        2020-02-01  1355.620642  1488.429810  
20003   2020-01-01   981.221823  1090.595219  
...                         ...          ...  
21266   2020-02-01     0.130282     0.171689  
21267   2020-01-01     0.071711     0.095030  
        2020-02-01     0.084321     0.115738  
21276   2020-01-01     0.028621     0.037913  
        2020-02-01     0.033912     0.046214  

[1560 rows x 10 columns]

In [41]:
# paso a formato Polars, teniendo en cuenta el indice
tb_forecast = pl.from_pandas(tb_forecast.reset_index())

In [42]:
# me quedo con ls predicciones de febrero-2020
# en 'mean' esta la prediccion, mas alla de los n-tiles
tb_final = tb_forecast.filter(pl.col("timestamp") ==  datetime(2020, 2, 1)).select(["item_id","mean"])

display(tb_final)

item_id,mean
i64,f64
20001,1297.394839
20002,1090.900843
20003,707.227027
20004,509.500685
20005,510.297339
…,…
21263,0.036944
21265,0.044744
21266,0.050203


## 1.5 Submit a Kaggle

In [43]:
# cambio nombre de campos a los que reconoce Kaggle
tb_final = tb_final.rename({
  "item_id": "product_id",
  "mean": "tn",
})

In [44]:
# Submit a Kaggle
if not empiojar:
  archivo= "AutoGluon_" + global_eval_metric + ".csv"
  mensaje= "AutoGluon " + global_eval_metric
else:
  archivo= "AutoGluon_empiojado_" + global_eval_metric + ".csv"
  mensaje= "AutoGluon logEMPIOJADO " + global_eval_metric + " al " + str(empiojar_ruido)

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje )